# Download Videos from S3

In [8]:
! pip install boto3
! pip install python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
import boto3
import os
from dotenv import load_dotenv
import pandas as pd


load_dotenv()

s3_bucket = 'deliberation-lab-recordings-exaptation'
s3_region = "us-east-1"
# recordings_folder = 'recordings'
recordings_folder = f"/Volumes/TalkLab_Recordings/{s3_bucket}" # where to store the recordings locally


s3 = boto3.client('s3', region_name=s3_region)

In [10]:
# download every recording from the s3 bucket

def list_all_s3_objects(bucket_name, prefix=''):
    paginator = s3.get_paginator('list_objects_v2')
    
    # Create a PageIterator from the Paginator
    page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
    
    all_objects = []
    
    for page in page_iterator:
        if 'Contents' in page:
            for obj in page['Contents']:
                all_objects.append(obj)
    
    return all_objects

# # without pagination just get the first 1000 objects
# response = s3.list_objects_v2(Bucket=s3_bucket)
# listings = response['Contents']

listings = list_all_s3_objects(s3_bucket)
len(listings)



5475

In [13]:
# to_download = [item for item in listings if item["LastModified"] > pd.Timestamp('2025-09-13', tz='UTC') and "pilot" in item["Key"]]
to_download = [item for item in listings if item["LastModified"] > pd.Timestamp('2026-01-01', tz='UTC') ]
len(to_download)

2484

In [15]:
# for every file in the s3 bucket that starts with deliberation/2024
# if the directory it is in doesn't already exist, create a folder in the recordings folder
# if the file iteself doesn't exist, download the file from the s3 bucket to the folder

for obj in to_download:
    if obj['Key'].startswith('deliberation/'):
        folder, file = obj['Key'].replace('deliberation/', "").split('/')
        if not os.path.exists(f'{recordings_folder}/{folder}'):
            print(f'Creating folder {folder}')
            os.makedirs(f'{recordings_folder}/{folder}')
        if not os.path.exists(f'{recordings_folder}/{folder}/{file}.webm'):
            print(f'Downloading {obj["Key"]}')
            s3.download_file(s3_bucket, obj['Key'], f'{recordings_folder}/{folder}/{file}.webm')

Creating folder 20260106_2205_pilot2SFPGH
Creating folder 20260106_2205_pilot7S6A88
Creating folder 20260106_2205_pilot8XX3B0
Creating folder 20260106_2205_pilot9MVJ2Z
Creating folder 20260106_2205_pilotB78GBJ
Creating folder 20260106_2205_pilotCHGNHQ
Creating folder 20260106_2205_pilotD98JZ6
Creating folder 20260106_2205_pilotG2P3DH
Creating folder 20260106_2205_pilotH72RRV
Creating folder 20260106_2205_pilotHNH7HQ
Creating folder 20260106_2205_pilotJBMYEM
Creating folder 20260106_2205_pilotKH7MPC
Creating folder 20260106_2205_pilotKWKG74
Creating folder 20260106_2205_pilotMQFV0D
Creating folder 20260106_2205_pilotN7AGC8
Creating folder 20260106_2205_pilotPWNDE7
Creating folder 20260106_2205_pilotRRAMXF
Creating folder 20260106_2205_pilotTA6VS9
Creating folder 20260106_2205_pilotVD27WJ
Creating folder 20260106_2205_pilotW2DVRS
Creating folder 20260106_2205_pilotZSQXS5
Creating folder 20260107_1620_pilot38DFFW
Creating folder 20260107_1620_pilotG96VJ7
Creating folder 20260107_1620_pilo

In [ ]:
# # figure out how much of the discussion was recorded

# data = pd.read_csv("../data/formatted_data.csv")

# for i, participant in data.iterrows():
#     recordingTimes = []
#     for recordingId in participant['recordingIds']:
        


In [ ]:
# ! ffprobe recordings/20240920_1732_CT_topMWAGR8/1726855356783-207f4a62-94d6-43e0-844c-2f8442334cb2-cam-audio-1726855372130.webm

In [ ]:
# ! ffmpeg -i recordings/20240920_1732_CT_topMWAGR8/1726855356783-207f4a62-94d6-43e0-844c-2f8442334cb2-cam-audio-1726855372130.webm 2>&1 | grep "Duration"

# Measure the quantity of video we have for each respondent
We want to check and see how much video we have, accounting for reloads and technical issues. Ideally, the measure would be the number of unique seconds in which we have at least 1 (10?) frames of video. Not quite sure how to do this yet.

In [ ]:
# ! ffmpeg -i recordings/20240920_1732_CT_topMWAGR8/1726855356783-207f4a62-94d6-43e0-844c-2f8442334cb2-cam-audio-1726855372130.webm -vf "showinfo" -f null - 2>&1 | grep showinfo